<a href="https://colab.research.google.com/github/Ayseatci/DI725_Assignment1/blob/dev/notebooks/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from google.colab import userdata

!git clone -b dev https://github.com/Ayseatci/DI725_Assignment1.git

%cd DI725_Assignment1

Cloning into 'DI725_Assignment1'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 97 (delta 46), reused 57 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 943.54 KiB | 14.30 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/DI725_Assignment1/DI725_Assignment1


## Preprocessing train data

In [3]:
# Loading train data
df = pd.read_csv("data/raw/train.csv")
removed_rows = df[~df["conversation"].str.contains(r'(?i)customer\s*:', na=False)]
print(f"Sample of removed conversations:\n{removed_rows['conversation'].head()}")

df = df[df["conversation"].str.contains(r'(?i)customer\s*:', na=False)].copy()


# Dropping NaNs
categorical_cols = ["issue_area", "issue_category", "product_category", "agent_experience_level"]
df = df.dropna(subset=["conversation", "customer_sentiment"] + categorical_cols).copy()

# Text preprocessing and replacing tags for customer and agent
def preprocess_dialogue(text):
    text = str(text).strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"(?i)\bcustomer\s*:", "<cust> ", text)
    text = re.sub(r"(?i)\bagent\s*:", "<agent> ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n+", "\n", text).strip()

df["text"] = df["conversation"].apply(preprocess_dialogue)

sentiment_map = {"negative": 0, "neutral": 1, "positive": 2}

df["label"] = df["customer_sentiment"].map(sentiment_map)

Sample of removed conversations:
191    Agent: You're welcome, Jane. Have a great day!
286    Agent: You're welcome, Jane. Have a great day!
750    Agent: You're welcome, Jane. Have a great day!
Name: conversation, dtype: object


In [4]:
df

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,text,label
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or ...,Mobile Number and Email Verification -> Verifi...,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,1
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox customer...,<agent> Thank you for calling BrownBox custome...,1
2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to...,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,1
3,Login and Account,Login Issues and Error Messages,Error message regarding exceeded attempts to e...,Login Issues and Error Messages -> Error messa...,neutral,Appliances,Water Purifier,less,inexperienced,"may struggle with ambiguous queries, rely on c...","Customer: Hi, I am facing an issue while loggi...","<cust> Hi, I am facing an issue while logging ...",1
4,Order,Order Delivery Issues,Delivery not attempted again,Order Delivery Issues -> Delivery not attempte...,negative,Electronics,Bp Monitor,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for contacting BrownBox custo...,<agent> Thank you for contacting BrownBox cust...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
965,Cancellations and returns,Return and Exchange,Package open or tampered on delivery,Return and Exchange -> Package open or tampere...,negative,Electronics,Mobile,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
966,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Men/Women/Kids,Backpack,medium,junior,"handles customer inquiries independently, poss...","Customer: Hi, I received an email from BrownBo...","<cust> Hi, I received an email from BrownBox s...",1
967,Warranty,Warranty Terms and Changes,Warranty mismatch between the website and the ...,Warranty Terms and Changes -> Warranty mismatc...,negative,Appliances,Water Purifier,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
968,Cancellations and returns,Return and Exchange,Checking the status of a refund,Return and Exchange -> Checking the status of ...,neutral,Appliances,Wet Grinder,medium,junior,"handles customer inquiries independently, poss...","Customer: Hi, I would like to check the status...","<cust> Hi, I would like to check the status of...",1


The initial preprocessing step implements a customer presence filter, which detects dialogues that do not consist of a customer line in the conversation. These are removed from both the training and testing sets. This step prevents the neutral bias and helps the model to interpret emotion rather than its ability to handle noise.

The pipeline also extracts quantitative  features to capture emotional signals. The frequency of customer exclamation marks and total conversation length is calculated before the text processing. To ensure these numeric features do not vanish the text embeddings during backpropagation, standardization is applied. The scaling step prevents larger raw values from dominating the weight updates and supports both faster convergence and balanced learning.

A normalization process to prepare the text for the Transformer's self-attention mechanism is applied to filtered conversation. The dialogue is standardized by replacing "Customer:" and "Agent:" identifiers with special tokens <cust> and <agent> to provide structural boundaries to the model. This allows the RoBERTa layers to distinguish between the source of the sentiment and the conetext of the interaction. Whitespace and line breaks is collapsed so that the input sequence is further optimized. This ensures that the model allocates its limited positional embeddings to meaningful vocabulary.


Furthermore, sentiment categories are mapped to ids so they can be utilized by the model.



In [5]:
df.columns

Index(['issue_area', 'issue_category', 'issue_sub_category',
       'issue_category_sub_category', 'customer_sentiment', 'product_category',
       'product_sub_category', 'issue_complexity', 'agent_experience_level',
       'agent_experience_level_desc', 'conversation', 'text', 'label'],
      dtype='object')

In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_path = "data/preprocessed/preprocessed_train.csv"
val_path = "data/preprocessed/preprocessed_val.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

# Auth and Setup
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "ayseatci00@gmail.com"
!git config --global user.name "Ayseatci"
!git remote set-url origin https://{token}@github.com/Ayseatci/DI725_Assignment1.git

!git add {train_path} {val_path}
!git commit -m "Add stratified preprocessed train and validation datasets"
!git push origin dev

[dev 09d7950] Add stratified preprocessed train and validation datasets
 2 files changed, 38354 insertions(+), 38353 deletions(-)
 create mode 100644 data/preprocessed/preprocessed_val.csv
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 23.31 KiB | 194.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/Ayseatci/DI725_Assignment1.git
   f633982..09d7950  dev -> dev


In [8]:
val_df[val_df["customer_sentiment"] == "positive"]

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,text,label
958,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Appliances,Kitchen Chimney,medium,experienced,"confidently handles complex customer issues, e...","Customer: Hi, I'm calling because I am interes...","<cust> Hi, I'm calling because I am interested...",2
182,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Men/Women/Kids,T-Shirt,less,junior,"handles customer inquiries independently, poss...","Agent: Hello, thank you for calling BrownBox c...","<agent> Hello, thank you for calling BrownBox ...",2
250,Order,Product Information and Tags,Finding seller's returns policy,Product Information and Tags -> Finding seller...,positive,Men/Women/Kids,Sweatshirt,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,2


In [9]:
removed_rows

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
191,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,Shorts,less,junior,"handles customer inquiries independently, poss...","Agent: You're welcome, Jane. Have a great day!"
286,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,T-Shirt,less,inexperienced,"may struggle with ambiguous queries, rely on c...","Agent: You're welcome, Jane. Have a great day!"
750,Order,Order Delivery Issues,Order approved but not shipped,Order Delivery Issues -> Order approved but no...,negative,Appliances,Sandwich Maker,less,experienced,"confidently handles complex customer issues, e...","Agent: You're welcome, Jane. Have a great day!"


The removed rows that do not include customer lines are displayed.

## Preprocessing test data

In [10]:
# Loading test data
df = pd.read_csv("data/raw/test.csv")
removed_rows = df[~df["conversation"].str.contains(r'(?i)customer\s*:', na=False)]
print(f"Sample of removed conversations:\n{removed_rows['conversation'].head()}")

df = df[df["conversation"].str.contains(r'(?i)customer\s*:', na=False)].copy()

# Dropping NaNs
categorical_cols = ["issue_area", "issue_category", "product_category", "agent_experience_level"]
df = df.dropna(subset=["conversation", "customer_sentiment"] + categorical_cols).copy()


# Text preprocessing and replacing tags for customer and agent
def preprocess_dialogue(text):
    text = str(text).strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"(?i)\bcustomer\s*:", "<cust> ", text)
    text = re.sub(r"(?i)\bagent\s*:", "<agent> ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n+", "\n", text).strip()

df["text"] = df["conversation"].apply(preprocess_dialogue)

df["label"] = df["customer_sentiment"].map(sentiment_map)

Sample of removed conversations:
Series([], Name: conversation, dtype: object)


In [11]:
df.count()

,0
issue_area,30
issue_category,30
issue_sub_category,30
issue_category_sub_category,30
customer_sentiment,30
product_category,30
product_sub_category,30
issue_complexity,30
agent_experience_level,30
agent_experience_level_desc,30


In [12]:
df.head()

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,text,label
0,Shopping,Pricing and Discounts,Discounts through exchange offers,Pricing and Discounts -> Discounts through exc...,negative,Appliances,Hand Blender,less,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
1,Login and Account,Account Reactivation and Deactivation,Reactivating an inactive account,Account Reactivation and Deactivation -> React...,negative,Men/Women/Kids,Wrist Watch,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
2,Cancellations and returns,Cash on Delivery (CoD) Refunds,Refund timelines for Cash on Delivery returns,Cash on Delivery (CoD) Refunds -> Refund timel...,negative,Appliances,Induction Cooktop,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
3,Order,Order Delivery Issues,Package shows as delivered but cannot be found,Order Delivery Issues -> Package shows as deli...,negative,Men/Women/Kids,Sunglas,high,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0
4,Cancellations and returns,Pickup and Shipping,Reimbursement of courier charges for return,Pickup and Shipping -> Reimbursement of courie...,negative,Electronics,Computer Monitor,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,<agent> Thank you for calling BrownBox Custome...,0


The  preprocessing steps applied to training data are done on the test dataset.

In [13]:
# Pushing preprocessed data under preprocessed file in GitHub repository
file_path = "data/preprocessed/preprocessed_test.csv"
df.to_csv(file_path, index=False)

# Auth and Push
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "ayseatci00@gmail.com"
!git config --global user.name "Ayseatci"
!git remote set-url origin https://{token}@github.com/Ayseatci/DI725_Assignment1.git

!git add {file_path}
!git commit -m "Updated preprocessed test data to be used for Roberta model"
!git push origin +dev

On branch dev
Your branch is up to date with 'origin/dev'.

nothing to commit, working tree clean
Everything up-to-date
